# EDA — équité et variables substituts

**Session de travail du 29 août 2026.** Troisième carnet exploratoire.

Le genre est **exclu de mon modèle** : il ne sert qu'à l'audit d'équité conduit
après coup. Encore faut-il que cette exclusion produise l'effet attendu. Ce
carnet répond à trois questions :

> 1. Comment la féminisation se distribue-t-elle dans le catalogue ?
> 2. À formation égale, l'admission est-elle équitable entre femmes et hommes ?
> 3. Quelles variables autorisées permettraient à mon modèle de **reconstituer**
>    le genre sans qu'il lui soit donné ?

La troisième est celle qui compte. Retirer une variable ne suffit pas à rendre un
modèle aveugle à ce qu'elle mesure : encore faut-il vérifier qu'aucune autre ne
la reconstitue.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

sys.path.insert(0, str(Path.cwd().parent / "src"))
from edumatch.config import load_settings

settings = load_settings("prod")
RAW = settings.raw_dir / "parcoursup"

df = pl.read_csv(
    RAW / "parcoursup_2025.csv",
    separator=";",
    encoding="utf8-lossy",
    infer_schema_length=2000,
)
print("session 2025 :", df.shape)

## 1. Un pourcentage sans son dénominateur n'est pas une information

Avant de décrire la féminisation, je dois trancher un point de méthode. La
colonne `pct_f` donne la part de femmes parmi les admis. Que vaut-elle pour une
formation qui n'admet personne ?

In [ ]:
sans_admis = (df["acc_tot"] == 0).sum()
print(f"formations sans aucun admis : {sans_admis}")
print(f"  dont pct_f enregistré à 0 % : {df.filter(pl.col('acc_tot') == 0).filter(pl.col('pct_f') == 0).height}")
print()
print(f"formations à moins de 20 % de femmes, sans filtre : {(df['pct_f'] < 20).sum()}")
print(f"formations à moins de 20 % de femmes, admis > 0   : {df.filter(pl.col('acc_tot') > 0).filter(pl.col('pct_f') < 20).height}")

**Ce que j'en conclus, et c'est une correction que j'apporte à mes propres
chiffres.** 179 formations n'admettent aucun candidat. Pour elles, `pct_f` vaut
mécaniquement 0 % — zéro femme sur zéro admis.

Ces 179 formations ne sont donc pas des formations qui admettent très peu de
femmes : **ce sont des formations qui n'admettent personne**. Les compter parmi
les formations peu féminisées gonfle le constat de 2 775 à 2 954.

Je retiens **2 775 formations, soit 19,7 %**, et j'écarte systématiquement les
formations sans admis du reste de cette analyse. Le principe est général et vaut
pour tout ratio : un pourcentage doit toujours être lu avec son dénominateur —
0 % sur 340 admis et 0 % sur 0 admis sont deux affirmations sans rapport.

In [ ]:
d = df.filter(pl.col("acc_tot") > 0)
f = d["pct_f"]

print(f"formations retenues : {d.height}")
print()
print(f"moins de 20 % de femmes : {(f < 20).sum():>5}  ({(f < 20).mean() * 100:>4.1f} %)")
print(f"entre 40 et 60 %        : {((f >= 40) & (f <= 60)).sum():>5}  ({((f >= 40) & (f <= 60)).mean() * 100:>4.1f} %)")
print(f"plus de 80 % de femmes  : {(f > 80).sum():>5}  ({(f > 80).mean() * 100:>4.1f} %)")
print()
total, total_f = d["acc_tot"].sum(), d["acc_tot_f"].sum()
print(f"part de femmes parmi l'ensemble des admis : {total_f / total * 100:.1f} %  ({total_f:,} / {total:,})")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(f.to_list(), bins=50, color="#4C72B0", edgecolor="white", linewidth=0.4)
ax.axvspan(0, 20, color="#C44E52", alpha=0.12)
ax.axvspan(80, 100, color="#C44E52", alpha=0.12)
ax.axvline(56.3, color="#55A868", linewidth=1.6, linestyle="--", label="56,3 % — part globale de femmes admises")
ax.set_xlabel("part de femmes parmi les admis (%)")
ax.set_ylabel("nombre de formations")
ax.set_title("La féminisation est polarisée, non centrée — session 2025")
ax.legend()
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

**Ce que j'en conclus.** La distribution n'a rien d'une cloche centrée sur la
moyenne. Elle est **polarisée** : 19,7 % des formations comptent moins de 20 % de
femmes, 17,9 % en comptent plus de 80 %, et seulement 21,5 % se situent dans la
zone équilibrée des 40 à 60 %.

Autrement dit, une formation prise au hasard a plus de chances d'être fortement
déséquilibrée dans un sens ou dans l'autre que d'être mixte.

La moyenne globale — 56,3 % de femmes parmi l'ensemble des admis — masque
entièrement ce phénomène. C'est un cas d'école : **résumer une distribution
bimodale par sa moyenne, c'est décrire une réalité qui n'existe nulle part.**

## 2. À formation égale, l'admission est-elle équitable ?

La polarisation précédente pourrait s'expliquer de deux façons très
différentes : soit les formations n'admettent pas les femmes et les hommes dans
les mêmes proportions, soit les candidatures elles-mêmes sont déjà polarisées en
amont. Ces deux hypothèses n'appellent pas du tout les mêmes conclusions, et je
peux les départager.

Je compare le taux d'admission des femmes et celui des hommes **à l'intérieur de
chaque formation**, en ne retenant que celles où les deux effectifs sont
suffisants pour que la comparaison ait un sens.

Une réserve de méthode : le fichier ne fournit pas de propositions ventilées par
sexe, seulement des admis. Cette comparaison porte donc sur les **admis**, et non
sur les propositions qui servent de base à mon label. Les deux quantités ne sont
pas identiques, et je le signale plutôt que de les confondre.

In [ ]:
apparie = (
    df.with_columns(
        [
            (pl.col("voe_tot") - pl.col("voe_tot_f")).alias("voe_h"),
            (pl.col("acc_tot") - pl.col("acc_tot_f")).alias("acc_h"),
        ]
    )
    .filter((pl.col("voe_tot_f") >= 30) & (pl.col("voe_h") >= 30))
    .with_columns(
        [
            (pl.col("acc_tot_f") / pl.col("voe_tot_f")).alias("taux_f"),
            (pl.col("acc_h") / pl.col("voe_h")).alias("taux_h"),
        ]
    )
    .with_columns((pl.col("taux_f") - pl.col("taux_h")).alias("ecart"))
)

print(f"formations avec au moins 30 vœux de chaque sexe : {apparie.height}")
e = apparie["ecart"]
print()
print("écart de taux d'admission, femmes moins hommes, dans la même formation")
print(f"  moyenne  {e.mean():>+8.4f}")
print(f"  médiane  {e.median():>+8.4f}")
print(f"  écart absolu inférieur à 5 points : {(e.abs() < 0.05).sum():>6}  ({(e.abs() < 0.05).mean() * 100:.1f} %)")
print(f"  femmes avantagées : {(e > 0).sum():>6}  ({(e > 0).mean() * 100:.1f} %)")
print(f"  hommes avantagés  : {(e < 0).sum():>6}  ({(e < 0).mean() * 100:.1f} %)")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(e.to_list(), bins=70, range=(-0.4, 0.4), color="#4C72B0", edgecolor="white", linewidth=0.4)
ax.axvline(0, color="#C44E52", linewidth=1.4)
ax.set_xlabel("taux d'admission des femmes − taux d'admission des hommes, même formation")
ax.set_ylabel("nombre de formations")
ax.set_title("À formation égale, l'écart d'admission entre sexes est centré sur zéro — session 2025")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

**Ce que j'en conclus, et c'est le résultat le plus contre-intuitif du projet.**

Sur 11 099 formations comparables, l'écart médian d'admission entre femmes et
hommes est de **−0,04 point**, c'est-à-dire nul. L'écart absolu reste inférieur à
5 points dans **83,8 %** des cas, et la répartition est presque parfaitement
symétrique : les femmes sont avantagées dans 48,8 % des formations, les hommes
dans 50,8 %.

**L'inégalité ne se joue donc pas à l'admission, mais en amont, dans le choix des
formations.** Les formations à 90 % de femmes ne sont pas des formations qui
écartent les hommes : ce sont des formations où les hommes ne candidatent pas.

Cela oriente entièrement ce que mon système peut prétendre faire. Il ne corrigera
pas une discrimination à l'entrée, puisque je n'en observe pas au niveau agrégé.
Il peut en revanche agir sur l'**information disponible au moment du choix** — ce
qui est précisément là où le déséquilibre se forme.

Je ne conclus pas pour autant à l'absence de discrimination : une médiane nulle
au niveau agrégé n'exclut pas des écarts marqués sur des sous-populations
particulières. C'est justement l'objet de l'audit d'équité, conduit plus tard sur
les prédictions du modèle et non sur les données brutes.

## 3. Quelles variables reconstituent le genre ?

C'est la question décisive. Exclure le genre du modèle ne sert à rien si une
autre variable le reconstitue — c'est ce qu'on appelle une **variable
substitut**.

Je mesure, pour chaque variable candidate, la part de la variance de la
féminisation qu'elle explique.

**Avec une précaution indispensable** : cette mesure est mécaniquement gonflée
par le nombre de modalités. Une variable à 4 000 modalités sur 14 000
observations « expliquerait » beaucoup de variance même en étant purement
aléatoire. Je mesure donc aussi le **niveau de hasard**, en recalculant la même
statistique après avoir mélangé la variable expliquée, et je ne retiens que
l'écart entre les deux.

In [ ]:
d = df.filter(pl.col("acc_tot") > 0)
y = d["pct_f"].to_numpy()
rng = np.random.default_rng(42)


def variance_expliquee(modalites: np.ndarray, valeurs: np.ndarray) -> float:
    """Part de la variance de `valeurs` expliquée par les groupes de `modalites`."""
    table = (
        pl.DataFrame({"groupe": modalites, "y": valeurs})
        .group_by("groupe")
        .agg(pl.len().alias("n"), pl.col("y").mean().alias("moyenne"))
    )
    inter = sum(n * (m - valeurs.mean()) ** 2 for _, n, m in table.iter_rows())
    return inter / len(valeurs) / valeurs.var()


candidates = ["fili", "select_form", "acad_mies", "dep", "ville_etab", "cod_uai"]
resultats = []
for col in candidates:
    modalites = d[col].to_numpy()
    observe = variance_expliquee(modalites, y)
    hasard = np.mean([variance_expliquee(modalites, rng.permutation(y)) for _ in range(5)])
    resultats.append((col, d[col].n_unique(), observe, hasard, observe - hasard))

print(f"{'variable':<16}{'modalités':>11}{'observé':>10}{'hasard':>9}{'net':>9}")
for col, modalites, observe, hasard, net in resultats:
    print(f"{col:<16}{modalites:>11}{observe * 100:>9.1f}%{hasard * 100:>8.1f}%{net * 100:>8.1f}%")

In [ ]:
resultats_tries = sorted(resultats, key=lambda r: r[4])
fig, ax = plt.subplots(figsize=(9, 4))
libelles = [r[0] for r in resultats_tries]
nets = [r[4] * 100 for r in resultats_tries]
hasards = [r[3] * 100 for r in resultats_tries]

ax.barh(libelles, nets, color="#C44E52", label="pouvoir explicatif net")
ax.barh(libelles, hasards, left=nets, color="#CCCCCC", label="part imputable au hasard")
ax.set_xlabel("part de la variance de la féminisation expliquée (%)")
ax.set_title("Variables substituts du genre, corrigées du nombre de modalités — session 2025")
ax.legend()
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

**Ce que j'en conclus, et c'est ce qui structurera mon audit d'équité.**

**D'abord, la correction change le classement.** `cod_uai` semblait expliquer
57,5 % de la féminisation ; une fois retiré le hasard imputable à ses 4 035
modalités, il n'en explique réellement que **28,9 %**. La moitié du résultat brut
était un artefact. Sans cette vérification, j'aurais inscrit dans mon dossier un
chiffre deux fois trop grand — et un lecteur refaisant le calcul l'aurait vu
immédiatement.

**Ensuite, le résultat contredit mon hypothèse de départ.** J'avais désigné
l'académie comme substitut à surveiller : elle n'explique que **1,4 %** de la
féminisation. Ce n'est pas un substitut sérieux. Les véritables substituts sont
l'établissement (28,9 %), la **filière** (19,5 %) et la ville (10,2 %).

**Enfin, et c'est le point difficile** : le deuxième substitut le plus puissant
est la filière — c'est-à-dire la variable la plus légitime et la plus
indispensable de mon modèle. Je ne peux pas la retirer sans détruire l'objet même
du système, qui est de comparer des formations.

J'en tire une conséquence que je dois assumer explicitement :

> **Mon modèle reconstituera partiellement le genre, quoi que je fasse.** Exclure
> la variable de genre est nécessaire, mais ne suffit pas à rendre le modèle
> aveugle au genre.

C'est pourquoi mon dispositif de non-discrimination ne peut pas reposer sur la
seule exclusion. Il repose sur trois niveaux :

1. **l'exclusion** de la variable de genre à l'entrée — nécessaire, insuffisante ;
2. **la mesure**, ici, de ce que les variables autorisées en reconstituent ;
3. **l'audit a posteriori** sur les prédictions du modèle, seul capable de
   révéler ce que les deux premiers niveaux laissent passer.

Une variable comme la ville, dont le pouvoir explicatif net atteint 10,2 % pour
un apport métier faible, est en revanche une candidate sérieuse à l'exclusion :
c'est un arbitrage que je tranche au moment de décider des variables retenues.

## Bilan

| Question | Ce que j'ai établi |
|---|---|
| Féminisation | polarisée : 19,7 % des formations sous 20 % de femmes, 17,9 % au-dessus de 80 %, 21,5 % seulement entre 40 et 60 % |
| Correction apportée | 179 formations sans aucun admis étaient comptées comme non féminisées — le constat passe de 2 954 à **2 775** |
| Équité à l'admission | écart médian **nul** entre femmes et hommes à formation égale ; inférieur à 5 points dans 83,8 % des cas |
| Où se joue l'inégalité | **en amont du choix**, pas à l'admission |
| Substituts réels | établissement 28,9 % · **filière 19,5 %** · ville 10,2 % · académie 1,4 % |
| Conséquence | l'exclusion du genre est nécessaire mais insuffisante : la filière, indispensable, en reconstitue une part |

**La suite** : la stabilité des séries entre 2018 et 2025, pour distinguer ce qui
relève d'une évolution réelle de ce qui relève d'un changement de règle de
publication.